# Advanced Models: Random Forest with Hyperparameter Tuning

This notebook implements a Random Forest Regressor with:
- K-Fold Cross Validation
- Hyperparameter Tuning using GridSearchCV
- Model Evaluation with R², Adjusted R², RMSE, MSE, MAE

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import joblib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
np.random.seed(42)

In [2]:
# ── Load Data ─────────────────────────────────────────────────────────────────
DATA_DIR = Path(r"C:\dev\Paddy_Prediction\data\final")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test  = pd.read_csv(DATA_DIR / "X_test.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").values.ravel()
y_test  = pd.read_csv(DATA_DIR / "y_test.csv").values.ravel()

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

X_train : (2231, 10)
X_test  : (558, 10)
y_train : (2231,)
y_test  : (558,)


## 1. Random Forest Regressor — GridSearchCV with K-Fold CV

Random Forest is an ensemble method that builds multiple decision trees and averages their predictions to reduce overfitting.

In [3]:
# ── Hyperparameter Grid ───────────────────────────────────────────────────────
rf_param_grid = {
    'n_estimators':      [100, 200, 300],      # Number of trees
    'max_depth':         [10, 20, 30, None],   # Max tree depth (None = unlimited)
    'min_samples_split': [2, 5, 10],           # Min samples to split a node
    'min_samples_leaf':  [1, 2, 4],            # Min samples at a leaf node
    'max_features':      ['sqrt', 'log2']      # Features considered per split
}

# ── K-Fold Cross Validation ───────────────────────────────────────────────────
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# ── GridSearchCV ──────────────────────────────────────────────────────────────
rf_grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=rf_param_grid,
    cv=kfold,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("Random Forest — Hyperparameter Tuning (GridSearchCV)")
rf_grid_search.fit(X_train, y_train)

best_rf_model = rf_grid_search.best_estimator_

print("\nRandom Forest — Best Results")
print(f"Best Parameters : {rf_grid_search.best_params_}")
print(f"Best CV MSE     : {-rf_grid_search.best_score_:.4f}")
print(f"Best Model      : {best_rf_model}")

Random Forest — Hyperparameter Tuning (GridSearchCV)
Fitting 5 folds for each of 216 candidates, totalling 1080 fits

Random Forest — Best Results
Best Parameters : {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best CV MSE     : 697202.8064
Best Model      : RandomForestRegressor(max_depth=10, max_features='sqrt', n_estimators=300,
                      random_state=42)


## 2. Model Evaluation

In [4]:
# ── Helper: Adjusted R² ───────────────────────────────────────────────────────
def adjusted_r2(r2, n, k):
    """Adjusted R² penalises extra features that don't improve the model.
    Args:
        r2 : standard R² score
        n  : number of samples
        k  : number of features
    """
    return 1 - (1 - r2) * (n - 1) / (n - k - 1)


# ── Evaluate on Train and Test ────────────────────────────────────────────────
n_features = X_train.shape[1]
results = {}

for split_name, X, y in [('Train', X_train, y_train), ('Test', X_test, y_test)]:
    preds = best_rf_model.predict(X)
    r2    = r2_score(y, preds)
    mse   = mean_squared_error(y, preds)
    results[split_name] = {
        'R² Score':    round(r2, 4),
        'Adjusted R²': round(adjusted_r2(r2, len(y), n_features), 4),
        'RMSE':        round(np.sqrt(mse), 4),
        'MSE':         round(mse, 4),
        'MAE':         round(mean_absolute_error(y, preds), 4)
    }

performance_df = pd.DataFrame(results).T
print("=== Random Forest Performance ===")
display(performance_df)

=== Random Forest Performance ===


,R² Score,Adjusted R²,RMSE,MSE,MAE
Train,0.9919,0.9918,832.8802,693689.3808,580.6327
Test,0.9911,0.9909,850.4634,723287.9858,593.3191


###Feature Importance

In [5]:
# ── Feature Importance Table ──────────────────────────────────────────────────
feat_imp = (
    pd.DataFrame({'Feature': X_train.columns, 'Importance': best_rf_model.feature_importances_})
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

print("=== Top 10 Features ===")
display(feat_imp.head(10))

=== Top 10 Features ===


,Feature,Importance
0,LP_nurseryarea(in Tonnes),0.118060
1,DAP_20days,0.113057
2,Micronutrients_70Days,0.109052
3,Pest_60Day(in ml),0.101940
4,Urea_40Days,0.100465
5,Potassh_50Days,0.098228
6,Nursery area (Cents),0.097987
7,Seedrate(in Kg),0.097745
8,LP_Mainfield(in Tonnes),0.096521
9,Weed28D_thiobencarb,0.066946


## 4. Save Model

In [6]:
model_dir = Path('../models')
model_dir.mkdir(parents=True, exist_ok=True)

rf_model_path = model_dir / 'random_forest_best.pkl'
joblib.dump(best_rf_model, rf_model_path)

print(f"Random Forest model saved to: {rf_model_path}")

Random Forest model saved to: ..\models\random_forest_best.pkl
